In [1]:
import csv
import pandas as pd
import numpy as np
from dateutil import parser

pd.options.mode.chained_assignment = None  # default='warn'

In [2]:
dataframe = pd.read_csv('TSLA_News_cleaned.csv')               # formatting date
 
df = dataframe[['Date', 'cleaned Title']]

def format_date(x):
    try:
        d = parser.parse(x)
        return d.strftime('%Y-%m-%d')
    except:
        return np.nan
    
df['Date'] = df['Date'].apply(lambda x: format_date(x))


df.dropna(inplace = True)                                     # remove dates that cannot be formatted 

df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'TSLA_News_cleaned.csv'

In [ ]:
df['text'] = df.groupby(['Date'])['cleaned Title'].transform(lambda x: ','.join(x))  # merging titles together based on dates
df.drop(columns = ['cleaned Title'], inplace = True)
df.drop_duplicates(inplace = True)

df.head()

# Sentiment analysis - vader

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()                                           # applying vader to text
df['compound'] = [analyzer.polarity_scores(x)['compound'] for x in df['text']]
df['neg'] = [analyzer.polarity_scores(x)['neg'] for x in df['text']]
df['neu'] = [analyzer.polarity_scores(x)['neu'] for x in df['text']]
df['pos'] = [analyzer.polarity_scores(x)['pos'] for x in df['text']]

df.drop(columns = ['neg', 'neu', 'pos'], inplace = True)                          # optional drops

df.head()

In [ ]:
# import flair
# from flair.models import TextClassifier
# from flair.data import Sentence

# sentiment_model = flair.models.TextClassififer.load('en-sentiment')

# df['compound'] = [classifier.predict(Sentence(x)).labels for x in df['text']]

In [ ]:
# !pip install flair

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])                             # sorting dates
df = df.sort_values(by='Date')

In [ ]:
df.to_csv('aapl_sentiment.csv')                                     # save to csv

# yfinance stock data

In [ ]:
from pandas_datareader import data

stock = 'TSLA'
start_date = '2021-01-01'
end_date = '2021-09-30'


panel_data = data.DataReader(stock, 'yahoo', start_date, end_date)
panel_data = panel_data.loc['2021-01-01':'2021-09-30']

panel_data.reset_index(inplace = True)

In [ ]:
panel_data['close_pct'] = panel_data['Close'].pct_change()

panel_data['difference'] = [1 if x >= 0 else 0 for x in panel_data.close_pct]

panel_data.head()

# merging both dataframes

In [ ]:
combined = pd.merge(panel_data, df, on='Date')

combined.head()

In [ ]:
combined['sentiment'] = [1 if x >= 0 else 0 for x in combined.compound]

combined

In [ ]:
# combined.corr()

In [ ]:
combined[combined.difference == combined.sentiment].shape[0] / combined.shape[0]

In [ ]:
# x = combined[combined.difference == 1]
# x[combined.sentiment == 1].shape[0]

In [ ]:
# x = combined[combined.difference == -1]
# x[combined.sentiment == -1].shape[0]

In [ ]:
import matplotlib.pyplot as plt

# fig, ax = plt.subplots(figsize=(16,9))
close_price = combined['close_pct']
sentiment = combined['compound']
# ax.plot(close_price, label = 'close price')

#plot 100 day MA on same plot
close_price = close_price.rolling(window=100).mean()
sentiment = sentiment.rolling(window=100).mean()

# ax.plot(close_price, label='close')
# ax.plot(sentiment, label='sentiment')

# ax.legend()

close_price.plot(figsize=(16,9))


In [ ]:
sentiment.plot(figsize=(16,9))